In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import os

In [77]:
csvs = []
for k in ["k4"]:
    for i in range(1,12):
        path = f"./{k}/peer{i}/peer{i}_extract_push.csv"
        # print(path)
        df = pd.read_csv(path, index_col=False)
        df['epoch'] = df['p_time_from'].str.split().str[2].astype(float)
        csvs.append(df)

In [78]:
df = pd.concat(csvs, ignore_index=True)
df['objsize'] = df['objsize'].fillna(0.0)

In [86]:
df = df.sort_values("epoch")
df = df.reset_index()

In [87]:
def toBytes(x):
    if not isinstance(x, str):
        return 0.0
    if "KiB" in x:
        objspl = x.split(" ")
        nobj = float(objspl[0]) * 1000
        return nobj
    elif "MiB" in x:
        objspl = x.split(" ")
        nobj = float(objspl[0]) * 1000000
        return nobj
    elif len(x) == len("92097d55807612bc90f62dc819ad7a3994c8e57e"):
        return 0.0
    else:
        objspl = x.split(" ")
        return float(objspl[0])
    return 0.0
df['objsize'] = df['objsize'].apply(toBytes)

In [88]:
df.shape

(3334, 15)

In [89]:
df.head()

,level_0,index,p_time_from,from,to,treeId,objsize,total_objs,delta_objs,old_c,new_c,p_time_to,merge_time,member_size,epoch
0,0,0,2025-11-28 12:41:19 1764313879195.248906000,peer1,peer2,6e00b9d335f90826f22c3923389dba71bf5e4be1,0.0,20.0,10.0,0000000000000000000000000000000000000000,376f766fcb7fccd2a5a82d08c188b844dec4ca09,2025-11-28 12:41:19 1764313879185.530368000,0.157488,3.0,1.764314e+12
1,1,1,2025-11-28 12:41:19 1764313879763.678912000,peer1,peer9,6e00b9d335f90826f22c3923389dba71bf5e4be1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.764314e+12
2,2,2,2025-11-28 12:41:20 1764313880336.596884000,peer1,peer2,6e00b9d335f90826f22c3923389dba71bf5e4be1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.764314e+12
3,3,3,2025-11-28 12:41:20 1764313880896.726022000,peer1,peer9,6e00b9d335f90826f22c3923389dba71bf5e4be1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.764314e+12
4,4,288,2025-11-28 12:41:20 1764313880999.014405000,peer2,peer10,6e00b9d335f90826f22c3923389dba71bf5e4be1,0.0,21.0,10.0,0000000000000000000000000000000000000000,b7f40b1ae3c57cb4a5024e7afd0971d256aec341,2025-11-28 12:41:20 1764313880988.842701000,0.043616,3.0,1.764314e+12


In [90]:
tree_pkv = {}
# tree_plist = []
start_uxt = df['epoch'][0] + 5000

for index, row in df.iterrows():
    pf = row['from']
    pt = row['to']
    treeId = row['treeId']
    
    if row['epoch'] < start_uxt:    
        if not treeId in tree_pkv:
            tree_pkv[treeId] = set()
        tree_pkv[treeId].add(pt)
        tree_pkv[treeId].add(pf) 
    else:
        # print(len(tree_pkv))
        # print(tree_pkv)
        tset = set()
        tc = 0
        for k,v in tree_pkv.items():
            tset.update(v)
            tc += 11-len(v)
        tc /= len(tree_pkv)
        # print(tset)
        print(tc, len(tset)-tc, tc/len(tset), len(tree_pkv))
        print()
        tree_pkv.clear() # clear the dict..
        if not treeId in tree_pkv:
            tree_pkv[treeId] = set()
        tree_pkv[treeId].add(pt)
        tree_pkv[treeId].add(pf)
        start_uxt += 5000
    # if row['epoch'] > 1764307489613.705793000:
    #     break

3.0 5.0 0.375 1

0.0 11.0 0.0 1

0.0 11.0 0.0 1

5.0 6.0 0.45454545454545453 3

7.4 3.5999999999999996 0.6727272727272727 10

8.0 3.0 0.7272727272727273 13

8.285714285714286 2.7142857142857135 0.7532467532467533 14

8.066666666666666 2.9333333333333336 0.7333333333333333 15

8.6875 2.3125 0.7897727272727273 16

8.428571428571429 2.571428571428571 0.7662337662337663 14

8.23076923076923 2.76923076923077 0.7482517482517482 13

7.923076923076923 3.0769230769230766 0.7202797202797203 13

8.23076923076923 2.76923076923077 0.7482517482517482 13

7.7 3.3 0.7000000000000001 10

8.25 2.75 0.75 12

8.333333333333334 2.666666666666666 0.7575757575757577 12

8.181818181818182 2.8181818181818183 0.743801652892562 11

7.583333333333333 3.416666666666667 0.6893939393939393 12

8.0 3.0 0.7272727272727273 9

8.0 3.0 0.7272727272727273 12

8.0 3.0 0.7272727272727273 9

8.714285714285714 2.2857142857142865 0.7922077922077921 14

8.68421052631579 2.3157894736842106 0.7894736842105263 19

8.35714285714285

In [9]:
# perkv = {}
# for i in range(1, 12):
#     perkv[f'peer{i}'] = set()
# uniq_trees = set()
# per_set = set()

# start_uxt = df['epoch'][0] + 5000
# max_hs = 0
# rounds = 1
# for index, row in df.iterrows():
#     if row['epoch'] < start_uxt:
#         pf = row['from']
#         pt = row['to']
#         per_set.add(pt)
#         per_set.add(pf)
#         uniq_trees.add(row['treeId'])

#         perkv[pf].add(row['treeId'])
#         max_hs = len(perkv[pf]) if len(perkv[pf]) > max_hs else max_hs
#         perkv[pt].add(row['treeId']) 
#         max_hs = len(perkv[pt]) if len(perkv[pt]) > max_hs else max_hs
#         # print(perkv)
#     else:
#         tc = 0
#         # print(f"---------round {rounds}----------")
#         # rounds += 1
#         # for (k,v) in perkv.items():
#         #     tc = tc + len(v)/max_hs
#         #     print(k, v)
#         # print()
#         # print(len(perkv)-tc)
            
#         per_set.clear()
#         start_uxt += 5000
#         pf = row['from']
#         pt = row['to']
#         per_set.add(pt)
#         per_set.add(pf)
#         uniq_trees.add(row['treeId'])

#         perkv[pf].add(row['treeId'])
#         perkv[pt].add(row['treeId']) 
        
#     if row['epoch'] > 1764307489613.705793000:
#         break
#         # print("\n")
# # print(uniq_trees)